# 📓 Notebook 05 — Cloud, Container & Kubernetes

Container escape checks, image CVEs, IaC misconfig, secret scanning, K8s RBAC abuse, CI/CD secret theft.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

In [ ]:
import os
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("cache:", CACHE)
except Exception as e:
    print("Drive notice:", e)
    CACHE = '/content/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("local cache:", CACHE)


In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

In [ ]:
%%bash
if ollama list | grep -q "qwen2.5-coder:7b"; then
  echo "qwen2.5-coder:7b ready"
else
  ollama pull qwen2.5-coder:7b
fi
ollama list


In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
if [ ! -d /content/hexstrike ]; then
  git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
fi
cd /content/hexstrike
if ! curl -sf localhost:8888/health >/dev/null; then
  nohup python3 hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
  sleep 5
fi
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling || true
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity || true
echo toolbox-ready


In [ ]:
%%bash
if ! pgrep -f "cloudflared tunnel --url http://localhost:8888" >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null || true
  chmod +x /usr/local/bin/cloudflared 2>/dev/null || true
  nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
  sleep 10
fi
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1


In [ ]:
%%bash
if ! command -v kubectl >/dev/null; then
  curl -fsSL -o /usr/local/bin/kubectl "https://dl.k8s.io/release/v1.30.0/bin/linux/amd64/kubectl" && chmod +x /usr/local/bin/kubectl || true
fi
pip -q install detect-secrets 2>/dev/null || true
if ! command -v trivy >/dev/null; then
  wget -q https://github.com/aquasecurity/trivy/releases/latest/download/trivy_0.58.0_Linux-64bit.tar.gz -O /tmp/t.tgz && tar -xzf /tmp/t.tgz -C /usr/local/bin trivy 2>/dev/null || true
fi
if ! command -v kube-bench >/dev/null; then
  wget -q https://github.com/aquasecurity/kube-bench/releases/latest/download/kube-bench_0.10.5_linux_amd64.tar.gz -O /tmp/kb.tgz && tar -xzf /tmp/kb.tgz -C /usr/local/bin kube-bench 2>/dev/null || true
fi
if ! command -v gitleaks >/dev/null; then
  wget -q https://github.com/gitleaks/gitleaks/releases/latest/download/gitleaks_8.21.2_linux_x64.tar.gz -O /tmp/gl.tgz && tar -xzf /tmp/gl.tgz -C /usr/local/bin gitleaks 2>/dev/null || true
fi
echo "cloud tools ready"


In [ ]:
%%bash
# --- local k8s (k3s, no systemd) ---
curl -sfL https://get.k3s.io | INSTALL_K3S_SKIP_START=true sh - 2>/dev/null || true
nohup k3s server --disable traefik >/var/log/k3s.log 2>&1 &
sleep 20
export KUBECONFIG=/etc/rancher/k3s/k3s.yaml
kubectl get nodes 2>&1 | head || echo "k3s starting in background"


In [ ]:
%%bash
# --- targets ---
docker run -d --name wrongsecrets --network host jeroenwillemsen/wrongsecrets
docker run -d --name vulnerable-nginx --network host nginxdemos/hello
sleep 15
echo "wrongsecrets -> http://localhost:8080"
curl -s -o /dev/null -w "wrongsecrets:%{http_code}\n" http://localhost:8080

In [ ]:
# --- example attacks ---
!trivy fs --scanners vuln,secret,misconfig /content/hexstrike 2>/dev/null | head -30 || true
!kube-bench run --targets node 2>/dev/null | head -30 || true
!gitleaks detect -s /content -v 2>/dev/null | head -20 || true


In [ ]:
%%bash
export KUBECONFIG=/etc/rancher/k3s/k3s.yaml
curl -fsSL https://raw.githubusercontent.com/kubernetes-goat/kubernetes-goat/master/deploy-k8s.sh | bash 2>/dev/null | tail -20 || true
kubectl get pods -A 2>/dev/null | head -30 || echo "k8s ready"
